# KDD Stage 2 — Preprocessing

Handles missing data per the proposal's mitigation plan: interpolation for short gaps,
multi-source cross-referencing, and explicit missingness indicators as model features
(never silently imputing without telling the model something was missing).

In [ ]:
import pandas as pd
import numpy as np
import hashlib, json
from pathlib import Path

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

stations = pd.read_csv(RAW / "station_readings.csv", parse_dates=["date"])
land_cover = pd.read_csv(RAW / "sanlc_land_cover.csv")
print(stations.isna().sum())

In [ ]:
NUMERIC = ["rainfall_mm_24h", "river_level_m", "soil_moisture", "temperature_c", "humidity_pct"]

def clean_region(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("date").reset_index(drop=True)
    # explicit missingness indicators BEFORE imputing (multi-source cross-reference in a real
    # deployment would fill these from a neighbouring station first; here we interpolate directly)
    for col in NUMERIC:
        df[f"{col}_missing"] = df[col].isna().astype(int)
    # short gaps (<=3 days): linear interpolation. Longer gaps: leave NaN -> dropped downstream,
    # never silently filled with a guess for a long span.
    df[NUMERIC] = df[NUMERIC].interpolate(method="linear", limit=3, limit_direction="both")
    # physically impossible values clipped rather than dropped (sensor noise, not garbage rows)
    df["rainfall_mm_24h"] = df["rainfall_mm_24h"].clip(0, 500)
    df["soil_moisture"] = df["soil_moisture"].clip(0, 1)
    df["humidity_pct"] = df["humidity_pct"].clip(0, 100)
    return df

cleaned = stations.groupby("region_id", group_keys=False).apply(clean_region)
remaining_gaps = cleaned[NUMERIC].isna().sum().sum()
print("Remaining NaNs after interpolation (long gaps, dropped next):", remaining_gaps)
cleaned = cleaned.dropna(subset=NUMERIC).reset_index(drop=True)
cleaned.to_parquet(PROC / "station_readings_clean.parquet", index=False)
cleaned.head()

In [ ]:
land_cover.to_parquet(PROC / "land_cover_clean.parquet", index=False)

def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

checksums = {p.name: sha256_of(p) for p in PROC.glob("*") if p.is_file()}
(PROC / "CHECKSUMS.sha256.json").write_text(json.dumps(checksums, indent=2))
checksums